> **公开学习副本**：保留教学代码与解释，已清除运行输出、执行计数、嵌入附件及个人元数据。完整执行原件和H02交接包保留在本地/组内私有空间。不要把本副本误认为团队已验收的新数据版本。
>
> 数据来自Olist；基础教学材料参考[CercaTrovato/olist-repurchase-analysis](https://github.com/CercaTrovato/olist-repurchase-analysis)，保留上游代码与数据许可。仅在本地有原始数据时运行；输出可能包含逐行客户数据，保持在被Git忽略的outputs目录。源码中的定义及限制仍应阅读。
> 历史学习课：用于理解问题，不取代当前H01/H02。窗口对比只针对已开放的三期，不要用它探索待封存的最终标签。

# 从原始数据独立提出问题：先不继承已有项目定义

本练习的目标不是证明已有复购方案正确，而是学会自己提出问题、决定定义、选择方法。
已有方案中的复购目标、批准订单、六个月历史、90天未来均不是本练习的默认要求。
不需要丢掉前面学到的表结构知识；这里补的是“事实如何支持判断”的过程。

## 今天要完成的环节
读取未经资格筛选的原始数据 → 描述事实 → 找出疑问 → 提出业务问题 → 写出待决定事项。
今天不选择最终标签，也不训练模型；不能用历史结果倒推一个最漂亮的研究题目。

数据：小组仓库分发的Olist原始ZIP，许可CC BY-NC-SA 4.0。
https://github.com/CercaTrovato/olist-repurchase-analysis


In [ ]:
from pathlib import Path
from zipfile import ZipFile
import pandas as pd

DATA = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'data/olist-v2.zip').is_file()), Path.cwd()) / 'data' / 'olist-v2.zip'
assert DATA.exists(), '请在 olist_learning 文件夹运行'
def read(name):
    with ZipFile(DATA) as z:
        with z.open(name) as f:
            return pd.read_csv(f)

orders = read('olist_orders_dataset.csv')
items = read('olist_order_items_dataset.csv')
customers = read('olist_customers_dataset.csv')
reviews = read('olist_order_reviews_dataset.csv')
print('订单行数：', len(orders))
print('商品明细行数：', len(items))
print('不同买家标识：', customers['customer_unique_id'].nunique())
print('评价行数：', len(reviews))
print('被评价的不同订单数：', reviews['order_id'].nunique())


## 1. 区分记录状态与我们自己的研究定义
先不筛订单，只交叉统计：每种记录状态有多少订单？有多少具有批准或送达时间？
这一步只是观察字段，不决定哪个状态“有效”。


In [ ]:
price_by_order = items.groupby('order_id')['price'].sum().rename('item_price_sum')
audit = orders.merge(price_by_order, on='order_id', how='left', validate='one_to_one')
audit['has_approval_time'] = audit['order_approved_at'].notna()
audit['has_delivery_time'] = audit['order_delivered_customer_date'].notna()
audit['has_item_amount'] = audit['item_price_sum'].notna()
status_table = audit.groupby('order_status').agg(
    orders=('order_id', 'size'),
    with_approval_time=('has_approval_time', 'sum'),
    with_delivery_time=('has_delivery_time', 'sum'),
    with_item_amount=('has_item_amount', 'sum')
)
print(status_table.to_string())


**从结果自己推理：**

1. canceled状态有625笔，其中484笔存在批准时间。
2. delivered状态有96478笔，其中96464笔存在批准时间，即14笔缺少批准时间。
3. 因此，“有批准时间”既不等于“最终状态记录为已送达”，也不是所有研究通用的订单质量标准。

**尚不能直接决定：**批准过的取消订单该不该排除？
要先问研究什么：研究下单/批准行为、最终交付、取消风险，可能需要不同事件定义。
若预测发生在下单或批准时点，不能把当时未知的最终状态用于输入或偷偷按未来状态筛历史名单。
状态与时间不一致也需要核查，不能先认定某个字段绝对正确。


## 2. 金额不完整，不等于金额为零
我们按订单连接明细，但不删除连接不到明细的订单。检查已知金额和未知金额。


In [ ]:
print('连接不到明细金额的订单：', audit['item_price_sum'].isna().sum())
print('已知金额中，最小值：', audit['item_price_sum'].min())
print('已知金额中，小于等于0的数量：', audit['item_price_sum'].le(0).sum())
print('评价分数分布：')
print(reviews['review_score'].value_counts(dropna=False).sort_index().to_string())


**从结果推理：**775笔订单没有匹配到商品金额；已知金额都大于0。因此在本文件里，条件“商品金额>0”同时会排除未知金额订单，不只是排除零金额订单。
这是否合理，取决于问题。取消风险研究可能需要保留部分没有商品明细的订单；商品金额分析则需要说明金额覆盖范围。不能未经判断把775笔都填0或都删除。

评价表99224行只覆盖98673个不同订单，不能直接假定每单一条评价。若以后研究低评分，需先决定多条评价怎样处理。

## 3. 由事实提出问题，暂不指定算法

| 观察到的事实 | 候选业务问题 | 必须进一步检查 |
|---|---|---|
| 有买家标识和订单时间 | 客户是否再次购买？怎样识别回访客户？ | 事件定义、观察期限、买家覆盖、重复订单 |
| 有物流实际/预计时间 | 哪些订单需要延迟风险预警？ | “延迟”按日期还是时间戳、预测时点、可用特征 |
| 有评价分数及评论 | 哪些订单容易获得较低评分？ | 低分阈值、多评价、未评价订单的选择偏差 |
| 有订单状态与时间 | 取消订单与其他订单有什么差异？ | 状态语义、预测时点、历史状态是否可获得 |

这些事实只使问题值得考察，不证明哪个问题已经可预测。也可以先做描述分析，不强制使用预测模型。

## 4. 先回答业务用途，再写规则

请先选一个你真的能解释用途的问题。我们可以独立选择，不必沿用小组题目；若最终选复购，也要重新说明理由。

1. 谁会读结果？
2. 他想做什么决定？
3. 什么时候需要结果？
4. 什么事件算结果发生？
5. 当时能知道哪些字段？
6. 当前数据缺少哪些证据？

例如只说“预测复购”仍不够：研究购买意向、批准事件、最终交付，对订单资格的定义可能不同。

## 5. 决策记录：现在必须保留为未决定

| 事项 | 当前状态 |
|---|---|
| 最终业务问题 | 未决定；需明确用途并比较候选 |
| 分析单位 | 随问题确定，可能订单/买家/客户快照 |
| 订单资格 | 未决定；不默认“批准且金额为正” |
| 预测时点 | 未决定；决定可用信息边界 |
| 目标/标签 | 未决定；先定义事件，再编代码 |
| 历史与未来期限 | 未决定；没有默认6个月或90天 |
| 方法 | 先决定描述/预测；预测再判断分类/回归/时间等任务 |
| 评价标准 | 根据要支持的决策及错误代价确定 |

本阶段不以产出模型为完成标准，而以能把“事实、解释、假设、决定”分开为完成标准。
完整后续路径：选问题与用途 → 定义分析单位/预测时点/结果 → 可行性检查 → 清理关联 → EDA和特征 → 时间验证与baseline → 合适模型 → 误差与改进 → 商业解释。

## 当下只需想一个具体判断
若你研究“买家再次发起购买”，与研究“买家再次成功收到商品”，你认为应该使用同一个订单事件定义吗？请先讲业务含义，再决定字段和筛选代码。
